In [ ]:
!git clone https://github.com/ultralytics/yolov5.git


Cloning into 'yolov5'...
remote: Enumerating objects: 17120, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 17120 (delta 47), reused 22 (delta 18), pack-reused 17043 (from 2)
Receiving objects: 100% (17120/17120), 15.78 MiB | 22.41 MiB/s, done.
Resolving deltas: 100% (11746/11746), done.


In [ ]:
%cd yolov5

/content/yolov5


In [ ]:
from PIL import Image
import torch
from models.common import DetectMultiBackend
from utils.general import check_img_size
from utils.augmentations import letterbox
import numpy as np

from PIL import Image
import torch
from models.common import DetectMultiBackend
from utils.general import check_img_size
from utils.augmentations import letterbox
import numpy as np

def predict_image(image_path, weights, imgsz=224, device='cpu'):
    device = torch.device(device)

    model = DetectMultiBackend(weights, device=device)
    stride = model.stride
    imgsz = check_img_size(imgsz, s=stride)

    img = Image.open(image_path).convert('RGB')
    img = letterbox(np.array(img), imgsz, stride=stride, auto=True)[0]
    img = np.transpose(img, (2, 0, 1))
    img = np.ascontiguousarray(img).astype(np.float32) / 255.0

    img_tensor = torch.from_numpy(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        pred = model(img_tensor)
    class_index = torch.argmax(pred, dim=1).item()
    confidence = torch.softmax(pred, dim=1)[0, class_index].item()

    class_names = model.names
    class_label = class_names[class_index]

    return class_label, confidence


image_path = "/content/download.jpg"
weights_path = "/content/best_fold_1.pt"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

predicted_label, confidence_score = predict_image(image_path, weights_path, device=device)

categories = {
    0: 'Others',
    1: 'Honda',
    2: 'Mazda',
    3: 'Mitsubishi',
    4: 'Suzuki',
    5: 'Toyota',
    6: 'Hyundai',
    7: 'KIA',
    8: 'VinFast'
}

print(f"Predicted Label: {categories.get(int(predicted_label))}")
print(f"Confidence Score: {confidence_score:.2f}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Fusing layers... 
Model summary: 117 layers, 4178217 parameters, 0 gradients, 10.4 GFLOPs


Predicted Label: Toyota
Confidence Score: 0.98
